In [1]:
# cell 0

!pip install -q ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 4.7 MB/s eta 0:00:00


In [2]:
# cell 1
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
secret_label = "HF"
secret_value = UserSecretsClient().get_secret(secret_label)
# HF token from Kaggle secrets, if present
try:
    os.environ["HF_TOKEN"] = secret_value
    login(secret_value)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


# Phase 1 — Decoder-only Legal LM (from scratch)

**Goal:** Train a ~55M-parameter decoder-only Transformer on Indian legal text
(Acts / High Courts / Supreme Court / Tribunals), then generate 128-word
continuations for a 512-word prompt and submit to Codabench.

**Design principles:**
- PyTorch used only for tensors, autograd, `nn.Linear`/`nn.Embedding`, optimizers.
- Every architectural component (norm, RoPE, attention, MLP, block) is written by hand.
- **Registries** make the token-mixer and channel-mixer pluggable, so this first
  decoder-only paradigm can be replaced later (linear attention, SSM, MoE, …)
  with a single registry entry — no other code changes.
- Config is a serializable dataclass saved inside every checkpoint.
- Training-time data sampling is a pure function of `(seed, step)` → safe resume.

**Preset used here:** S (d_model=512, n_layer=12, n_head=8) ≈ 55M params.

In [3]:
# Cell 2 — Imports, paths, device, reproducibility

import os, sys, json, math, time, zlib, random, hashlib, gc, shutil, zipfile
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional, Dict, List, Tuple, Any
from collections import OrderedDict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ---- Paths ---------------------------------------------------------------
IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR  = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
INPUT_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Folder that contains bpe_tokenizer.py + artifacts (vocab.json, merges.txt, config.json)
TOKENIZER_DIR = INPUT_DIR
if not TOKENIZER_DIR.exists():
    TOKENIZER_DIR = Path(".").resolve()

print("WORK_DIR      :", WORK_DIR)
print("TOKENIZER_DIR :", TOKENIZER_DIR)

# ---- Device / precision --------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_FP16 = (device.type == "cuda") and not torch.cuda.is_bf16_supported()
print("Device:", device, "| fp16 autocast:", USE_FP16)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ---- Reproducibility -----------------------------------------------------
SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

WORK_DIR      : /kaggle/working
TOKENIZER_DIR : /kaggle/input/datasets/ascyrax/capstone-tokenizer-large
Device: cuda | fp16 autocast: False
GPU: Tesla T4


In [4]:
# cell 3

# --- Path to the dataset folder that contains bpe_tokenizer.py + artifacts ---
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large")

# Make sure Python can find bpe_tokenizer.py
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))

# The real function name (from the file you shared):
from bpe_tokenizer import BPETokenizer, load_tokenizer

# --- Load the tokenizer -------------------------------------------------------
tokenizer = load_tokenizer(str(TOKENIZER_DIR))

# VOCAB_SIZE = tokenizer.vocab_size
VOCAB_SIZE = len(tokenizer.id_to_bytes)  # ✅ Dynamically gets the real size
BOS_ID     = tokenizer.bos_id
EOS_ID     = tokenizer.eos_id
PAD_ID     = tokenizer.pad_id
UNK_ID     = tokenizer.unk_id

print("vocab_size:", VOCAB_SIZE)
print("BOS:", BOS_ID, "| EOS:", EOS_ID, "| PAD:", PAD_ID, "| UNK:", UNK_ID)

# --- Sanity check: round-trip -------------------------------------------------
_s = "The Supreme Court of India held that the appeal must be allowed."
_ids = tokenizer.encode(_s, allow_special=False)
_decoded = tokenizer.decode(_ids)
print("round-trip ok:", _decoded == _s)
print("ids:", _ids[:20], "...")

# --- Sanity check: offsets ----------------------------------------------------
_ids, offs = tokenizer.encode(_s, allow_special=False, return_offsets=True)
print("offset example:", _s[offs[0][0]:offs[0][1]], "->", _ids[0])

vocab_size: 32000
BOS: 2 | EOS: 3 | PAD: 0 | UNK: 1
round-trip ok: True
ids: [731, 1660, 544, 390, 927, 1095, 445, 382, 960, 1727, 422, 1314, 169] ...
offset example: The -> 731


In [5]:
#  CELL 4

@dataclass
class ModelConfig:
    # --- shape ---
    vocab_size: int = 32000
    n_layer: int = 12
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8                # set < n_head for GQA
    head_dim: int = 64
    ff_mult: float = 8/3              # SwiGLU default; 4.0 for GELU
    d_ff: Optional[int] = None        # if set, overrides ff_mult
    max_seq_len: int = 1024

    # --- components ---
    norm_type: str = "rmsnorm"        # "layernorm" | "rmsnorm"
    norm_eps: float = 1e-5
    pos_type: str = "rope"            # "rope" | "learned" | "none"
    rope_theta: float = 10000.0
    mlp_type: str = "swiglu"          # "swiglu" | "gelu"
    mixer_type: str = "attention"     # token mixer registry key
    channel_mixer_type: str = "mlp"   # channel mixer registry key

    # --- misc ---
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    causal: bool = True
    use_sdpa: bool = True
    grad_checkpointing: bool = False

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim, \
            f"d_model ({self.d_model}) must equal n_head*head_dim ({self.n_head*self.head_dim})"
        assert self.n_head % self.n_kv_head == 0, "n_head must be divisible by n_kv_head"
        if self.d_ff is None:
            # SwiGLU: hidden = (2/3) * ff_mult * d  (so 3 matrices give the same FLOPs)
            raw = int(self.ff_mult * self.d_model * 2 / 3 * 3 / 2)  # keep as-is
            # simpler: hidden = round to 64 multiple of (8/3)*d
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


# ---- Default presets -----------------------------------------------------
def preset_S(vocab_size=VOCAB_SIZE, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size, n_layer=12, d_model=512,
                       n_head=8, head_dim=64, max_seq_len=max_seq_len)

def preset_M(vocab_size=VOCAB_SIZE, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size, n_layer=12, d_model=768,
                       n_head=12, head_dim=64, max_seq_len=max_seq_len)

def preset_L(vocab_size=VOCAB_SIZE, max_seq_len=1024):
    return ModelConfig(vocab_size=vocab_size, n_layer=16, d_model=1024,
                       n_head=16, head_dim=64, max_seq_len=max_seq_len)

print(json.dumps(asdict(preset_S()), indent=2))

{
  "vocab_size": 32000,
  "n_layer": 12,
  "d_model": 512,
  "n_head": 8,
  "n_kv_head": 8,
  "head_dim": 64,
  "ff_mult": 2.6666666666666665,
  "d_ff": 1408,
  "max_seq_len": 1024,
  "norm_type": "rmsnorm",
  "norm_eps": 1e-05,
  "pos_type": "rope",
  "rope_theta": 10000.0,
  "mlp_type": "swiglu",
  "mixer_type": "attention",
  "channel_mixer_type": "mlp",
  "bias": false,
  "attn_dropout": 0.0,
  "resid_dropout": 0.0,
  "embd_dropout": 0.0,
  "tie_embeddings": true,
  "init_std": 0.02,
  "causal": true,
  "use_sdpa": true,
  "grad_checkpointing": false
}


In [6]:
# cell 5
class LayerNorm(nn.Module):
    """Classic LayerNorm with learnable gain and bias."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.b = nn.Parameter(torch.zeros(d))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var  = x.var(-1, keepdim=True, unbiased=False)
        x_hat = (x - mean) / torch.sqrt(var + self.eps)
        return x_hat * self.g + self.b


class RMSNorm(nn.Module):
    """RMSNorm: no mean centering, no bias. Cheaper, and works as well."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        # compute in fp32 for stability
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)
        return (x / rms) * self.g


def build_norm(cfg: ModelConfig):
    if cfg.norm_type == "layernorm":
        return LayerNorm(cfg.d_model, cfg.norm_eps)
    elif cfg.norm_type == "rmsnorm":
        return RMSNorm(cfg.d_model, cfg.norm_eps)
    raise ValueError(cfg.norm_type)

In [7]:
# cell 6

class RoPE(nn.Module):
    """
    Precomputes cos/sin tables for RoPE up to `max_seq_len`.
    Applies pairwise rotation to Q and K. Applied in fp32 for stability.
    """
    def __init__(self, head_dim: int, max_seq_len: int, theta: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        self.head_dim = head_dim
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.outer(t, inv_freq)          # [max_seq_len, head_dim/2]
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    @staticmethod
    def _rotate(x, cos, sin):
        """x: [B, H, T, D]; cos,sin: [B, 1, T, D/2]."""
        x1 = x[..., 0::2]
        x2 = x[..., 1::2]
        o1 = x1 * cos - x2 * sin
        o2 = x1 * sin + x2 * cos
        return torch.stack([o1, o2], dim=-1).flatten(-2)

    def forward(self, q, k, position_ids=None):
        """
        q, k: [B, H, T, D]
        position_ids: [B, T] (optional); if None, uses 0..T-1
        Returns rotated q, k.
        """
        B, H, T, D = q.shape
        if position_ids is None:
            cos = self.cos[:T]                                # [T, D/2]
            sin = self.sin[:T]
            cos = cos.unsqueeze(0).unsqueeze(0)               # [1,1,T,D/2]
            sin = sin.unsqueeze(0).unsqueeze(0)
        else:
            # gather per position
            cos = self.cos[position_ids]                      # [B, T, D/2]
            sin = self.sin[position_ids]
            cos = cos.unsqueeze(1)                            # [B,1,T,D/2]
            sin = sin.unsqueeze(1)

        # upcast to fp32 for numeric stability, cast back
        in_dtype = q.dtype
        q = q.float(); k = k.float()
        q = self._rotate(q, cos.float(), sin.float())
        k = self._rotate(k, cos.float(), sin.float())
        return q.to(in_dtype), k.to(in_dtype)

In [8]:
# cell 7

class Attention(nn.Module):
    """
    Multi-head / grouped-query attention.
    - Causal self-attention when `context is None`.
    - Cross-attention when `context` is provided (K,V come from context).
    - Optional KV cache for autoregressive decoding.
    - Optional padding mask + explicit position ids.
    """
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.n_head    = cfg.n_head
        self.n_kv_head = cfg.n_kv_head
        self.head_dim  = cfg.head_dim
        self.scale     = 1.0 / math.sqrt(self.head_dim)

        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head    * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head    * cfg.head_dim, cfg.d_model,   bias=cfg.bias)

        self.attn_dropout = nn.Dropout(cfg.attn_dropout)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, n_head):
        B, T, _ = x.shape
        return x.view(B, T, n_head, self.head_dim).transpose(1, 2)   # [B,H,T,D]

    def forward(self, x, rope=None, position_ids=None, attention_mask=None,
                kv_cache=None, context=None):
        """
        x: [B, T, d_model]
        attention_mask: [B, S] with 1=keep, 0=pad  (S = full key length incl. cache)
        kv_cache: optional tuple (k_prev, v_prev) each [B, n_kv_head, S_prev, D]
        context: [B, S, d_model] for cross-attention
        Returns: (out, new_kv_cache)
        """
        B, T, _ = x.shape
        kv_src = context if context is not None else x
        S = kv_src.shape[1]

        q = self._split(self.q_proj(x), self.n_head)                  # [B, H, T, D]
        k = self._split(self.k_proj(kv_src), self.n_kv_head)          # [B,Hkv,S,D]
        v = self._split(self.v_proj(kv_src), self.n_kv_head)

        # ---- RoPE on Q,K (only for self-attention) ----
        if rope is not None:
            if context is None:
                q, k = rope(q, k, position_ids=position_ids)
            else:
                # cross-attention: no RoPE (context is not positional)
                pass

        # ---- KV cache concatenation ----
        if kv_cache is not None:
            k_prev, v_prev = kv_cache
            k = torch.cat([k_prev, k], dim=2)
            v = torch.cat([v_prev, v], dim=2)
        new_cache = (k, v)

        # ---- GQA: repeat K,V heads to match Q ----
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k = k.repeat_interleave(rep, dim=1)
            v = v.repeat_interleave(rep, dim=1)

        # ---- Build attention mask ----
        # `is_causal` only makes sense when the current query block is the
        # *last* T positions of the full S. For cached decoding T=1, S=S_prev+1.
        is_causal = (context is None) and (self.cfg.causal is True) and (T > 1 or kv_cache is None)
        # For cached decode with T==1 we don't need a causal mask (only one query).

        # Additive mask from padding (broadcast to [B,1,T,S])
        pad_mask = None
        if attention_mask is not None:
            # attention_mask: [B, S] with 1=keep, 0=pad
            keep = attention_mask[:, None, None, :].to(torch.bool)     # [B,1,1,S]
            pad_mask = torch.zeros(B, 1, T, S, device=x.device, dtype=torch.float32)
            pad_mask = pad_mask.masked_fill(~keep, float("-inf"))

        # ---- Compute attention ----
        use_sdpa = self.cfg.use_sdpa and hasattr(F, "scaled_dot_product_attention")

        if use_sdpa and pad_mask is None and (is_causal or T == 1):
            out = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=None,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=is_causal,
            )
        elif use_sdpa:
            # combine causal + padding via additive mask
            if is_causal:
                causal = torch.triu(
                    torch.full((T, S), float("-inf"), device=x.device, dtype=torch.float32),
                    diagonal=S - T + 1,
                )
                additive = causal.view(1, 1, T, S)
                if pad_mask is not None:
                    additive = additive + pad_mask
            else:
                additive = pad_mask
            out = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=additive,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=False,
            )
        else:
            # ---- Manual path ----
            scores = (q @ k.transpose(-2, -1)) * self.scale            # [B,H,T,S]
            if is_causal:
                causal = torch.triu(
                    torch.ones(T, S, device=x.device, dtype=torch.bool),
                    diagonal=S - T + 1,
                )
                scores = scores.masked_fill(causal, float("-inf"))
            if pad_mask is not None:
                scores = scores + pad_mask.to(scores.dtype)
            scores = F.softmax(scores.float(), dim=-1).to(q.dtype)
            scores = self.attn_dropout(scores)
            out = scores @ v

        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [9]:
# cell 8

class SwiGLU(nn.Module):
    """SwiGLU: down(silu(gate(x)) * up(x))."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj   = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GELUMLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        hidden = int(cfg.ff_mult * cfg.d_model) if cfg.d_ff is None else cfg.d_ff
        self.fc1 = nn.Linear(cfg.d_model, hidden, bias=cfg.bias)
        self.fc2 = nn.Linear(hidden, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)

    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))

In [10]:
# cell 9

# ---------- Registries: the extensibility hook ----------
TOKEN_MIXERS: Dict[str, callable] = {}
CHANNEL_MIXERS: Dict[str, callable] = {}

def register_token_mixer(name: str):
    def deco(cls):
        TOKEN_MIXERS[name] = cls
        return cls
    return deco

def register_channel_mixer(name: str):
    def deco(cls):
        CHANNEL_MIXERS[name] = cls
        return cls
    return deco


# Register the defaults. NOTE: attention needs the rope module, so the
# registry stores a factory that takes (cfg, rope) for token mixers.
@register_token_mixer("attention")
def _build_attention(cfg, rope):
    return Attention(cfg)

@register_channel_mixer("mlp")
def _build_mlp(cfg):
    if cfg.mlp_type == "swiglu":
        return SwiGLU(cfg)
    elif cfg.mlp_type == "gelu":
        return GELUMLP(cfg)
    raise ValueError(cfg.mlp_type)


def build_token_mixer(cfg, rope):
    if cfg.mixer_type not in TOKEN_MIXERS:
        raise KeyError(f"Unknown token mixer '{cfg.mixer_type}'. "
                       f"Available: {list(TOKEN_MIXERS)}")
    return TOKEN_MIXERS[cfg.mixer_type](cfg, rope)

def build_channel_mixer(cfg):
    if cfg.channel_mixer_type not in CHANNEL_MIXERS:
        raise KeyError(f"Unknown channel mixer '{cfg.channel_mixer_type}'. "
                       f"Available: {list(CHANNEL_MIXERS)}")
    return CHANNEL_MIXERS[cfg.channel_mixer_type](cfg)


class Block(nn.Module):
    """Pre-norm transformer block: x = x + mixer(norm(x)); x = x + cmix(norm(x))."""
    def __init__(self, cfg: ModelConfig, rope):
        super().__init__()
        self.norm1 = build_norm(cfg)
        self.norm2 = build_norm(cfg)
        self.mixer = build_token_mixer(cfg, rope)
        self.cmix  = build_channel_mixer(cfg)

    def forward(self, x, **kw):
        h, new_cache = self.mixer(self.norm1(x), **kw)
        x = x + h
        x = x + self.cmix(self.norm2(x))
        return x, new_cache

In [11]:
# cell 10

class Backbone(nn.Module):
    """Embedding → N Blocks → final norm. Returns hidden states [B,T,d]."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_drop = nn.Dropout(cfg.embd_dropout)

        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) \
            if cfg.pos_type == "rope" else None

        # learned absolute positions as an alternative
        if cfg.pos_type == "learned":
            self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model)
        else:
            self.pos_emb = None

        self.blocks = nn.ModuleList([Block(cfg, self.rope) for _ in range(cfg.n_layer)])
        self.norm_f = build_norm(cfg)

    def forward(self, input_ids, attention_mask=None, position_ids=None,
                kv_caches=None, return_hidden=True):
        B, T = input_ids.shape
        x = self.tok_emb(input_ids)

        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(T, device=input_ids.device).unsqueeze(0)
            x = x + self.pos_emb(position_ids)

        x = self.emb_drop(x)

        new_caches = []
        for i, blk in enumerate(self.blocks):
            cache_i = kv_caches[i] if kv_caches is not None else None
            x, nc = blk(
                x,
                position_ids=position_ids,
                attention_mask=attention_mask,
                kv_cache=cache_i,
            )
            new_caches.append(nc)

        x = self.norm_f(x)
        return x, new_caches


class LegalLM(nn.Module):
    """Backbone + tied LM head. Keeps the head separate so task heads can attach later."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.backbone = Backbone(cfg)
        self.lm_head  = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.backbone.tok_emb.weight

    def forward(self, input_ids, targets=None, attention_mask=None,
                position_ids=None, kv_caches=None, return_hidden=False):
        h, new_caches = self.backbone(
            input_ids, attention_mask=attention_mask,
            position_ids=position_ids, kv_caches=kv_caches,
        )
        logits = self.lm_head(h)

        loss = None
        if targets is not None:
            # cross-entropy in fp32 on upcast logits
            loss = F.cross_entropy(
                logits.float().view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-100,
            )

        if return_hidden:
            return logits, loss, h, new_caches
        return logits, loss, new_caches

In [12]:
# cell 11

def init_weights(model: nn.Module, cfg: ModelConfig):
    """
    - N(0, 0.02) for linear/embedding weights, zeros for biases.
    - Scale residual-branch output projections by 1/sqrt(2*n_layer).
    """
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=cfg.init_std)

    # residual scaling: o_proj in each attention, down_proj in each MLP
    scale = 1.0 / math.sqrt(2 * cfg.n_layer)
    for blk in model.backbone.blocks:
        # attention path
        if hasattr(blk.mixer, "o_proj"):
            with torch.no_grad():
                blk.mixer.o_proj.weight.mul_(scale)
        # channel mixer path
        if hasattr(blk.cmix, "down_proj"):
            with torch.no_grad():
                blk.cmix.down_proj.weight.mul_(scale)
        if hasattr(blk.cmix, "fc2"):
            with torch.no_grad():
                blk.cmix.fc2.weight.mul_(scale)

In [13]:
# cell 12

def count_params(model: LegalLM) -> Dict[str, int]:
    """Count tied embeddings once, assert total is in the 50M-300M window."""
    emb_params = model.backbone.tok_emb.weight.numel()
    seen = set()
    def _sum(mod):
        n = 0
        for p in mod.parameters():
            if id(p) in seen:
                continue
            seen.add(id(p))
            n += p.numel()
        return n
    total = _sum(model)
    non_emb = total - emb_params
    return {"total": total, "embedding": emb_params, "non_embedding": non_emb}


def build_param_groups(model: nn.Module, weight_decay: float):
    """Decay on 2D weight matrices; no decay on norm gains, biases, embeddings."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        is_emb = ("tok_emb" in name) or ("pos_emb" in name)
        is_norm = ("norm" in name) or (".g" in name) or (".b" in name)
        if p.ndim >= 2 and not is_emb and not is_norm:
            decay.append(p)
        else:
            no_decay.append(p)
    return [
        {"params": decay,    "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]


def save_model(path: Path, model: LegalLM, cfg: ModelConfig, extra: dict = None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "model": model.state_dict(),
        "config": asdict(cfg),
        "extra": extra or {},
    }, path)

def load_model(path: Path, map_location="cpu") -> Tuple[LegalLM, ModelConfig, dict]:
    ckpt = torch.load(path, map_location=map_location)
    cfg = ModelConfig(**ckpt["config"])
    model = LegalLM(cfg)
    model.load_state_dict(ckpt["model"])
    return model, cfg, ckpt.get("extra", {})

In [14]:
# cell 13

def run_model_tests(cfg: ModelConfig = None, verbose=True):
    cfg = cfg or preset_S()
    torch.manual_seed(0)
    model = LegalLM(cfg).to(device)
    init_weights(model, cfg)
    model.eval()

    B, T = 2, 32
    ids = torch.randint(0, cfg.vocab_size, (B, T), device=device)

    results = {}

    # 1) Parameter count
    pc = count_params(model)
    results["param_count"] = pc
    if verbose:
        print(f"[1] params: total={pc['total']/1e6:.2f}M "
              f"emb={pc['embedding']/1e6:.2f}M non_emb={pc['non_embedding']/1e6:.2f}M")
    assert 50_000_000 <= pc["total"] <= 300_000_000, "param count outside 50M-300M"

    # 2) Initial loss ~ ln(vocab)
    with torch.no_grad():
        _, loss, _ = model(ids, targets=ids)
    expected = math.log(cfg.vocab_size)
    if verbose:
        print(f"[2] initial loss: {loss.item():.4f}  expected ~{expected:.4f}")
    results["init_loss"] = loss.item()

    # 3) Causality — change a future token; earlier logits must not change
    with torch.no_grad():
        logits_a, _, _ = model(ids)
        ids_b = ids.clone()
        ids_b[:, -1] = (ids_b[:, -1] + 1) % cfg.vocab_size
        logits_b, _, _ = model(ids_b)
    delta = (logits_a[:, :-1] - logits_b[:, :-1]).abs().max().item()
    if verbose:
        print(f"[3] causality max |Δlogits| on earlier positions: {delta:.2e}")
    assert delta < 1e-4, "Causality violated"
    results["causality_delta"] = delta

    # 4) KV-cache equivalence — prefill + step
    with torch.no_grad():
        full_logits, _, _ = model(ids)                    # [B, T, V]
        # prefill
        logits_p, _, caches = model(ids[:, :1], return_hidden=False, kv_caches=None)
        # then decode one token at a time
        step_logits = [logits_p[:, 0:1, :]]
        caches = caches if not isinstance(caches, tuple) else caches
        cur_caches = caches
        for t in range(1, T):
            pos = torch.tensor([[t]], device=device)
            lg, _, cur_caches = model(ids[:, t:t+1], position_ids=pos, kv_caches=cur_caches)
            step_logits.append(lg)
        step_logits = torch.cat(step_logits, dim=1)
    kv_delta = (step_logits - full_logits).abs().max().item()
    if verbose:
        print(f"[4] KV-cache max |Δlogits|: {kv_delta:.2e}")
    assert kv_delta < 1e-3, "KV-cache mismatch"
    results["kv_delta"] = kv_delta

    # 5) SDPA vs manual attention
    cfg_manual = ModelConfig(**{**asdict(cfg), "use_sdpa": False})
    model_manual = LegalLM(cfg_manual).to(device)
    model_manual.load_state_dict(model.state_dict())
    model_manual.eval()
    with torch.no_grad():
        lg_manual, _, _ = model_manual(ids)
    sdpa_delta = (full_logits - lg_manual).abs().max().item()
    if verbose:
        print(f"[5] SDPA vs manual max |Δlogits|: {sdpa_delta:.2e}")
    assert sdpa_delta < 1e-3, "SDPA vs manual mismatch"
    results["sdpa_delta"] = sdpa_delta

    # 6) Padding equivalence (left-padded)
    seq = ids[0:1]  # [1, T]
    L = 6
    pad = torch.zeros(1, L, dtype=torch.long, device=device)
    padded = torch.cat([pad, seq], dim=1)             # [1, T+L]
    mask = torch.cat([torch.zeros(1, L, dtype=torch.long, device=device),
                      torch.ones(1, T,  dtype=torch.long, device=device)], dim=1)
    pos = torch.cat([torch.zeros(1, L, dtype=torch.long, device=device),
                     torch.arange(T, device=device).unsqueeze(0)], dim=1)
    with torch.no_grad():
        lg_pad, _, _ = model(padded, attention_mask=mask, position_ids=pos)
    # The last T positions should match the unpadded forward's logits.
    pad_delta = (lg_pad[0, L:] - full_logits[0]).abs().max().item()
    if verbose:
        print(f"[6] padding equivalence max |Δlogits|: {pad_delta:.2e}")
    assert pad_delta < 1e-3, "Padding equivalence failed"
    results["padding_delta"] = pad_delta

    # 7) Config round-trip
    tmp = WORK_DIR / "_test_ckpt.pt"
    save_model(tmp, model, cfg)
    model2, cfg2, _ = load_model(tmp, map_location=device)
    model2 = model2.to(device).eval()
    with torch.no_grad():
        lg2, _, _ = model2(ids)
    rt_delta = (lg2 - full_logits).abs().max().item()
    if verbose:
        print(f"[7] config round-trip max |Δlogits|: {rt_delta:.2e}")
    assert rt_delta < 1e-5, "Config round-trip mismatch"
    results["roundtrip_delta"] = rt_delta

    print("\nAll model tests PASSED ✓")
    return model, results


# Run on S preset
_ = run_model_tests(preset_S())

[1] params: total=54.93M emb=16.38M non_emb=38.55M
[2] initial loss: 8.9129  expected ~10.3735
[3] causality max |Δlogits| on earlier positions: 0.00e+00
[4] KV-cache max |Δlogits|: 2.50e-06
[5] SDPA vs manual max |Δlogits|: 1.91e-06
[6] padding equivalence max |Δlogits|: 2.50e-06
[7] config round-trip max |Δlogits|: 0.00e+00

All model tests PASSED ✓


In [15]:
# Cell 14 (v2) — Stream each JSON file from HF, memory-safe encoder

import os, json, time, hashlib
from pathlib import Path
from array import array
import ijson
from huggingface_hub import HfFileSystem
from tqdm.auto import tqdm
import numpy as np

HF_REPO = "Exploration-Lab/CS-781-Capstone"

SOURCE_PATHS = {
    "acts":          "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts":   "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals":     "Phase_1_Pretraining/Train/train_tribunals.json",
}

SHARD_DIR     = WORK_DIR / "shards"
SHARD_TOKENS  = 50_000_000      # smaller → lower peak memory
VAL_FRACTION  = 0.005
VAL_CAP_TOTAL = 5_000_000

TOKEN_CAPS = {
    "acts":          20_000_000,
    "high_courts":   65_000_000,
    "supreme_court": 25_000_000,
    "tribunals":     40_000_000,
}

def clean_text(t): return (t or "").replace("\r\n", "\n").strip()

def is_val(doc_id, salt="phase1"):
    h = int(hashlib.md5((salt + "::" + doc_id).encode()).hexdigest(), 16)
    return (h % 10_000) / 10_000.0 < VAL_FRACTION

def stream_docs(path_in_repo):
    """
    Stream docs from a HF-hosted JSON file without downloading the whole file.
    Assumes the file is a JSON array of objects (as the phase_1 files appear to be).
    """
    fs = HfFileSystem()
    url = f"datasets/{HF_REPO}/{path_in_repo}"
    with fs.open(url, "rb") as f:
        for i, doc in enumerate(ijson.items(f, "item")):
            if isinstance(doc, dict):
                doc.setdefault("_idx", i)
                yield doc


def encode_all():
    summary = {}
    for source, path_in_repo in SOURCE_PATHS.items():
        out_dir = SHARD_DIR / source
        out_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n[{source}] streaming {path_in_repo}")
        meta = {"source": source, "hf_path": path_in_repo, "shards": [],
                "total_tokens": 0, "total_bytes": 0, "num_docs": 0}

        shard_ids = array("H")     # uint16, 2 bytes per token
        shard_idx = 0
        val_ids   = array("H")
        val_cap   = VAL_CAP_TOTAL // len(SOURCE_PATHS)
        cap       = TOKEN_CAPS[source]

        def flush():
            nonlocal shard_ids, shard_idx
            if len(shard_ids) == 0:
                return
            arr = np.frombuffer(shard_ids, dtype=np.uint16)   # zero-copy view
            final = out_dir / f"shard_{shard_idx:04d}.bin"
            tmp   = final.with_suffix(".bin.tmp")
            arr.tofile(tmp); tmp.rename(final)
            meta["shards"].append({"name": final.name, "tokens": int(arr.size)})
            meta["total_tokens"] += int(arr.size)
            print(f"  [{source}] {final.name}: {arr.size/1e6:.1f}M tokens")
            shard_ids = array("H")    # fresh buffer
            shard_idx += 1

        pbar = tqdm(desc=f"  {source}", unit="doc")
        try:
            for doc in stream_docs(path_in_repo):
                pbar.update(1)
                text = clean_text(doc.get("text"))
                if not text:
                    continue
                doc_id = str(doc.get("url") or doc.get("id") or doc.get("_idx"))
                meta["num_docs"] += 1
                meta["total_bytes"] += len(text.encode("utf-8"))

                ids = tokenizer.encode(text, allow_special=False)

                if is_val(doc_id) and len(val_ids) < val_cap:
                    val_ids.append(BOS_ID)
                    val_ids.extend(ids)
                    val_ids.append(EOS_ID)
                    continue

                shard_ids.append(BOS_ID)
                shard_ids.extend(ids)
                shard_ids.append(EOS_ID)

                if len(shard_ids) >= SHARD_TOKENS:
                    flush()
                if meta["total_tokens"] + len(shard_ids) >= cap:
                    print(f"  [{source}] hit cap {cap/1e6:.0f}M")
                    break
        finally:
            pbar.close()

        flush()
        if len(val_ids) > 0:
            arr = np.frombuffer(val_ids, dtype=np.uint16)
            (out_dir / "val.bin").write_bytes(arr.tobytes())
            meta["val_tokens"] = int(arr.size)

        with open(out_dir / "meta.json", "w") as f:
            json.dump(meta, f, indent=2)
        summary[source] = meta
        print(f"[{source}] {meta['total_tokens']/1e6:.1f}M tokens, "
              f"{meta['num_docs']} docs, val={meta.get('val_tokens',0)/1e6:.2f}M\n")

    return summary


# ---- Smoke test on 1M per source first ----
# for k in TOKEN_CAPS: TOKEN_CAPS[k] = 1_000_000
results = encode_all()


[acts] streaming Phase_1_Pretraining/Train/train_acts.json


  acts: 0doc [00:00, ?doc/s]

  [acts] hit cap 20M
  [acts] shard_0000.bin: 20.0M tokens
[acts] 20.0M tokens, 2228 docs, val=0.07M


[high_courts] streaming Phase_1_Pretraining/Train/train_high_courts.json


  high_courts: 0doc [00:00, ?doc/s]

  [high_courts] shard_0000.bin: 50.0M tokens
  [high_courts] hit cap 65M
  [high_courts] shard_0001.bin: 15.0M tokens
[high_courts] 65.0M tokens, 112740 docs, val=0.31M


[supreme_court] streaming Phase_1_Pretraining/Train/train_supreme_court.json


  supreme_court: 0doc [00:00, ?doc/s]

  [supreme_court] hit cap 25M
  [supreme_court] shard_0000.bin: 25.0M tokens
[supreme_court] 25.0M tokens, 3650 docs, val=0.12M


[tribunals] streaming Phase_1_Pretraining/Train/train_tribunals.json


  tribunals: 0doc [00:00, ?doc/s]

  [tribunals] hit cap 40M
  [tribunals] shard_0000.bin: 40.0M tokens
[tribunals] 40.0M tokens, 8466 docs, val=0.20M



In [16]:
import json
total = 0
for s in ["acts", "high_courts", "supreme_court", "tribunals"]:
    p = WORK_DIR / "shards" / s / "meta.json"
    if not p.exists():
        print(f"{s:15s} MISSING"); continue
    m = json.loads(p.read_text())
    ntok = m["total_tokens"]
    total += ntok
    vtok = m.get("val_tokens", 0)
    print(f"{s:15s} train={ntok/1e6:7.2f}M  val={vtok/1e6:5.2f}M  "
          f"docs={m['num_docs']:>7,}  shards={len(m['shards'])}")
print(f"{'TOTAL':15s} train={total/1e6:7.2f}M")

# Disk size check
import subprocess
print("\nDisk usage:")
print(subprocess.run(["du", "-sh", str(WORK_DIR/"shards")],
                     capture_output=True, text=True).stdout)

acts            train=  20.01M  val= 0.07M  docs=  2,228  shards=1
high_courts     train=  65.00M  val= 0.31M  docs=112,740  shards=2
supreme_court   train=  25.00M  val= 0.12M  docs=  3,650  shards=1
tribunals       train=  40.00M  val= 0.20M  docs=  8,466  shards=1
TOTAL           train= 150.01M

Disk usage:
288M	/kaggle/working/shards



In [17]:
# cell 15

@dataclass
class MixConfig:
    source_weights: Dict[str, float]        # normalized
    temperature: float = 1.0                # alpha for size-based mixing

    @classmethod
    def from_sizes(cls, sizes: Dict[str, int], temperature: float = 1.0):
        # p_i ∝ size_i^alpha
        keys = list(sizes.keys())
        arr = np.array([sizes[k] for k in keys], dtype=np.float64)
        arr = np.power(arr, temperature)
        arr = arr / arr.sum()
        return cls(source_weights=dict(zip(keys, arr.tolist())),
                   temperature=temperature)


class ShardReader:
    """Memory-mapped shards for one source."""
    def __init__(self, source_dir: Path):
        self.dir = Path(source_dir)
        meta = json.loads((self.dir / "meta.json").read_text())
        self.shards = meta["shards"]
        self.total_tokens = sum(s["tokens"] for s in self.shards)
        self.arrays = [np.memmap(self.dir / s["name"], dtype=np.uint16, mode="r")
                       for s in self.shards]
        self.shard_weights = np.array([s["tokens"] for s in self.shards], dtype=np.float64)
        self.shard_weights /= self.shard_weights.sum()
        self.val_path = self.dir / "val.bin"
        self.val_arr = np.memmap(self.val_path, dtype=np.uint16, mode="r") \
            if self.val_path.exists() else None


class MixedWindowLoader:
    """
    Pure function of (seed, step). Resuming reproduces the same data order.
    Returns (input_ids, targets) of shape [B, T] and [B, T].
    """
    def __init__(self, sources: Dict[str, ShardReader], mix: MixConfig,
                 seq_len: int, micro_batch: int, seed: int = 0):
        self.sources = sources
        self.mix = mix
        self.seq_len = seq_len
        self.micro_batch = micro_batch
        self.seed = seed
        self.keys = list(sources.keys())
        self.p = np.array([mix.source_weights[k] for k in self.keys])

    def _rng(self, step: int) -> np.random.Generator:
        s = (self.seed * 1_000_003 + step * 2_654_435_761) & 0xFFFFFFFF
        return np.random.default_rng(s)

    def get_batch(self, step: int):
        rng = self._rng(step)
        B, T = self.micro_batch, self.seq_len
        inputs  = np.empty((B, T),     dtype=np.int64)
        targets = np.empty((B, T),     dtype=np.int64)
        for b in range(B):
            src = rng.choice(self.keys, p=self.p)
            reader = self.sources[src]
            shard_idx = rng.choice(len(reader.arrays), p=reader.shard_weights)
            arr = reader.arrays[shard_idx]
            max_start = max(1, len(arr) - (T + 1))
            start = int(rng.integers(0, max_start))
            window = np.asarray(arr[start:start + T + 1], dtype=np.int64)
            if window.size < T + 1:
                pad = np.zeros(T + 1 - window.size, dtype=np.int64)
                window = np.concatenate([window, pad])
            inputs[b]  = window[:-1]
            targets[b] = window[1:]
        return (torch.from_numpy(inputs), torch.from_numpy(targets))


class FixedValLoader:
    """Non-overlapping windows over val arrays; comparable across runs."""
    def __init__(self, sources: Dict[str, ShardReader], seq_len: int,
                 windows_per_source: int, batch_size: int = 8):
        self.seq_len = seq_len
        self.batch_size = batch_size
        self.windows = []   # list of (src, np.array)
        for name, reader in sources.items():
            if reader.val_arr is None:
                continue
            n = min(windows_per_source, len(reader.val_arr) // (seq_len + 1))
            for i in range(n):
                s = i * (seq_len + 1)
                w = np.asarray(reader.val_arr[s:s + seq_len + 1], dtype=np.int64)
                self.windows.append((name, w))

    def __len__(self):
        return (len(self.windows) + self.batch_size - 1) // self.batch_size

    def __iter__(self):
        for i in range(0, len(self.windows), self.batch_size):
            chunk = self.windows[i:i + self.batch_size]
            B = len(chunk)
            T = self.seq_len
            inputs  = np.zeros((B, T), dtype=np.int64)
            targets = np.zeros((B, T), dtype=np.int64)
            srcs = []
            for j, (name, w) in enumerate(chunk):
                inputs[j] = w[:-1]; targets[j] = w[1:]
                srcs.append(name)
            yield srcs, torch.from_numpy(inputs), torch.from_numpy(targets)

In [18]:
# cell 16

@dataclass
class TrainConfig:
    seq_len: int = 1024
    micro_batch: int = 8
    grad_accum: int = 16           # -> ~128k tokens / step
    peak_lr: float = 6e-4
    min_lr: float = 6e-5
    warmup_steps: int = 300
    total_steps: int = 5000
    weight_decay: float = 0.1
    betas: Tuple[float, float] = (0.9, 0.95)
    eps: float = 1e-8
    grad_clip: float = 1.0
    eval_every: int = 500
    eval_windows: int = 250        # ~1M tokens at seq_len=1024
    log_every: int = 50
    ckpt_every_minutes: float = 25
    time_budget_hours: float = 8.0
    seed: int = 1337
    output_dir: str = "runs/exp01"


def cosine_lr(step, warmup, total, peak, minimum):
    if step < warmup:
        return peak * (step + 1) / warmup
    if step >= total:
        return minimum
    prog = (step - warmup) / max(1, total - warmup)
    return minimum + 0.5 * (peak - minimum) * (1 + math.cos(math.pi * prog))


def evaluate(model, val_loader, device, use_fp16):
    model.eval()
    total_loss, total_tok = 0.0, 0
    per_src = {}
    with torch.no_grad():
        for srcs, x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_fp16):
                _, loss, _ = model(x, targets=y)
            n_tok = y.numel()
            total_loss += loss.item() * n_tok
            total_tok  += n_tok
            for s in set(srcs):
                # rough per-source bucket (windows may be batched across sources)
                per_src.setdefault(s, [0.0, 0])
                per_src[s][0] += loss.item() * n_tok / len(set(srcs))
                per_src[s][1] += n_tok // len(set(srcs))
    avg = total_loss / max(1, total_tok)
    per_src = {k: v[0] / max(1, v[1]) for k, v in per_src.items()}
    model.train()
    return avg, per_src


def train(cfg: ModelConfig, tcfg: TrainConfig, train_loader, val_loader,
          resume_from: Optional[Path] = None):
    out_dir = Path(tcfg.output_dir); out_dir.mkdir(parents=True, exist_ok=True)
    log_path = out_dir / "log.jsonl"

    model = LegalLM(cfg).to(device)
    if resume_from is None:
        init_weights(model, cfg)
    else:
        ckpt = torch.load(resume_from, map_location=device)
        model.load_state_dict(ckpt["model"])
        print(f"resumed from {resume_from} at step {ckpt['step']}")

    if cfg.grad_checkpointing:
        model.backbone.grad_checkpointing_enable = True  # hook for later

    opt = torch.optim.AdamW(
        build_param_groups(model, tcfg.weight_decay),
        lr=tcfg.peak_lr, betas=tcfg.betas, eps=tcfg.eps,
    )
    scaler = torch.amp.GradScaler('cuda', enabled=USE_FP16)

    start_step = 0
    if resume_from is not None:
        ckpt = torch.load(resume_from, map_location=device)
        opt.load_state_dict(ckpt["opt"])
        scaler.load_state_dict(ckpt["scaler"])
        start_step = ckpt["step"] + 1
        torch.set_rng_state(ckpt["rng"].cpu() if hasattr(ckpt["rng"], "cpu") else ckpt["rng"])
        if device.type == "cuda":
            torch.cuda.set_rng_state(ckpt["cuda_rng"])

    model.train()
    t0 = time.time()
    last_ckpt_t = time.time()
    tokens_seen = 0
    best_val = float("inf")

    for step in range(start_step, tcfg.total_steps):
        lr = cosine_lr(step, tcfg.warmup_steps, tcfg.total_steps, tcfg.peak_lr, tcfg.min_lr)
        for g in opt.param_groups:
            g["lr"] = lr

        opt.zero_grad(set_to_none=True)
        step_loss = 0.0
        for micro in range(tcfg.grad_accum):
            # unique, reproducible key per (optimizer step, micro step)
            x, y = train_loader.get_batch(step * tcfg.grad_accum + micro)
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_FP16):
                _, loss, _ = model(x, targets=y)
            scaler.scale(loss / tcfg.grad_accum).backward()
            step_loss += loss.item() / tcfg.grad_accum
            tokens_seen += y.numel()

        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), tcfg.grad_clip)
        scaler.step(opt); scaler.update()

        if step % tcfg.log_every == 0:
            dt = time.time() - t0
            tok_s = tokens_seen / max(1e-6, dt)
            rec = {
                "step": step, "loss": step_loss, "lr": lr,
                "grad_norm": float(gn), "tokens_seen": tokens_seen,
                "tokens_per_s": tok_s, "elapsed_s": dt,
                "scaler_scale": scaler.get_scale(),
            }
            with open(log_path, "a") as f:
                f.write(json.dumps(rec) + "\n")
            print(f"step {step:5d} | loss {step_loss:.4f} | lr {lr:.2e} "
                  f"| gnorm {gn:.2f} | {tok_s/1000:.1f}k tok/s | {dt/60:.1f}m")

        # ---- eval ----
        if step > 0 and step % tcfg.eval_every == 0:
            val_loss, per_src = evaluate(model, val_loader, device, USE_FP16)
            print(f"  [eval] step {step} val_loss={val_loss:.4f} "
                  f"ppl={math.exp(val_loss):.2f} per_src={ {k: round(v,3) for k,v in per_src.items()} }")
            with open(log_path, "a") as f:
                f.write(json.dumps({"step": step, "val_loss": val_loss,
                                    "val_per_src": per_src}) + "\n")
            if val_loss < best_val:
                best_val = val_loss
                save_model(out_dir / "best.pt", model, cfg,
                           extra={"step": step, "val_loss": val_loss})
            model.train()

        # ---- checkpoint ----
        if (time.time() - last_ckpt_t) / 60 >= tcfg.ckpt_every_minutes:
            ckpt = {
                "model": model.state_dict(),
                "opt": opt.state_dict(),
                "scaler": scaler.state_dict(),
                "step": step,
                "config": asdict(cfg),
                "train_config": asdict(tcfg),
                "rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state() if device.type == "cuda" else None,
            }
            torch.save(ckpt, out_dir / "last.pt")
            shutil.copy(out_dir / "last.pt", out_dir / "last_prev.pt")
            last_ckpt_t = time.time()
            print(f"  [ckpt] saved at step {step}")

        # ---- time budget ----
        if (time.time() - t0) / 3600 >= tcfg.time_budget_hours:
            print(f"time budget reached at step {step}")
            break

    # final save
    save_model(out_dir / "final.pt", model, cfg, extra={"step": step})
    return model, out_dir

In [19]:
# Cell 17 — Shard reader builder
SOURCES = ["acts", "high_courts", "supreme_court", "tribunals"]

def build_readers(shard_root: Path = SHARD_DIR):
    readers = {}
    for s in SOURCES:
        d = shard_root / s
        if (d / "meta.json").exists():
            readers[s] = ShardReader(d)
    return readers

In [20]:
# Cell 18 — Benchmark run (200 steps, ~5 min)
readers = build_readers()
print("readers:", {k: v.total_tokens for k, v in readers.items()})

mix = MixConfig.from_sizes({k: v.total_tokens for k, v in readers.items()})
print("mix:", mix.source_weights)

train_loader = MixedWindowLoader(readers, mix, seq_len=1024,
                                 micro_batch=8, seed=SEED)
val_loader   = FixedValLoader(readers, seq_len=1024, windows_per_source=64)

readers: {'acts': 20010586, 'high_courts': 65000132, 'supreme_court': 25002938, 'tribunals': 40000225}
mix: {'acts': 0.1333915626114626, 'high_courts': 0.4332941162958113, 'supreme_court': 0.1666708296147608, 'tribunals': 0.2666434914779653}


In [21]:
# # quick test run

# cfg  = preset_S(vocab_size=VOCAB_SIZE, max_seq_len=1024)
# tcfg = TrainConfig(
#     seq_len=1024,
#     micro_batch=32, grad_accum=4,
#     peak_lr=6e-4, min_lr=6e-5,
#     warmup_steps=10,
#     total_steps=50,                  # <-- tiny
#     eval_every=25,                   # 2 evals → best.pt gets written
#     log_every=5,
#     ckpt_every_minutes=999,          # disable time-based ckpt
#     time_budget_hours=0.2,           # hard cap: 12 min
#     output_dir=str(WORK_DIR / "runs" / "quick_test"),
# )
# model, run_dir = train(cfg, tcfg, train_loader, val_loader)

In [22]:
# Cell 19 — Full baseline run (around 9 hours)
cfg  = preset_S(vocab_size=VOCAB_SIZE, max_seq_len=1024)
tcfg = TrainConfig(
    seq_len=1024, 
    micro_batch=32, grad_accum=4,
    peak_lr=6e-4, min_lr=6e-5,
    warmup_steps=400, 
    total_steps=6000,
    eval_every=250, 
    log_every=50,
    ckpt_every_minutes=20, 
    time_budget_hours=9,
    output_dir=str(WORK_DIR / "runs" / "S_baseline"),
)
model, run_dir = train(cfg, tcfg, train_loader, val_loader)

step     0 | loss 10.4685 | lr 1.50e-06 | gnorm 12.70 | 7.9k tok/s | 0.1m
step    50 | loss 7.7846 | lr 7.65e-05 | gnorm 2.47 | 7.8k tok/s | 3.6m
step   100 | loss 5.3852 | lr 1.51e-04 | gnorm 2.53 | 7.7k tok/s | 7.1m
step   150 | loss 4.9100 | lr 2.26e-04 | gnorm 1.06 | 7.7k tok/s | 10.7m
step   200 | loss 4.4015 | lr 3.01e-04 | gnorm 1.33 | 7.7k tok/s | 14.3m
step   250 | loss 4.2336 | lr 3.76e-04 | gnorm 0.97 | 7.7k tok/s | 17.8m
  [eval] step 250 val_loss=4.4718 ppl=87.51 per_src={'acts': 4.817, 'high_courts': 3.544, 'supreme_court': 5.047, 'tribunals': 4.479}
  [ckpt] saved at step 278
step   300 | loss 4.0319 | lr 4.51e-04 | gnorm 0.80 | 7.6k tok/s | 21.6m
step   350 | loss 3.6238 | lr 5.26e-04 | gnorm 0.83 | 7.6k tok/s | 25.2m
step   400 | loss 3.7237 | lr 6.00e-04 | gnorm 0.95 | 7.6k tok/s | 28.8m
step   450 | loss 3.4919 | lr 6.00e-04 | gnorm 0.99 | 7.6k tok/s | 32.4m
step   500 | loss 3.7031 | lr 6.00e-04 | gnorm 0.72 | 7.6k tok/s | 36.0m
  [eval] step 500 val_loss=3.8052 ppl

In [23]:
# import os
# for f in ["best.pt", "final.pt", "log.jsonl"]:
#     p = WORK_DIR / "runs" / "quick_test" / f
#     print(f"{f:12s} {'OK' if p.exists() else 'MISSING':8s} "
#           f"{p.stat().st_size/1e6:.1f} MB" if p.exists() else "")

In [24]:
# # Load the quick checkpoint
# model_q, cfg_q, extra_q = load_model(WORK_DIR / "runs" / "quick_test" / "best.pt",
#                                      map_location=device)
# model_q = model_q.to(device).eval()
# print(f"loaded step {extra_q.get('step')}, val_loss {extra_q.get('val_loss')}")

# # Generate one continuation
# prompt = "The issue involved in these petitions is as to whether the acquisition proceedings can be challenged at a belated stage."
# prompt_ids = tokenizer.encode(prompt, allow_special=False)
# out_ids = generate(model_q, [prompt_ids], model_q.cfg,
#                    max_new_tokens=128, temperature=0.0,
#                    repetition_penalty=1.1, eos_id=EOS_ID)
# print("\n--- generated ---")
# print(tokenizer.decode(out_ids[0]))
# print("---")

In [25]:
# # Load only 10 test prompts to make it fast
# prompts = load_test_prompts_hf(max_n=10)
# print("loaded", len(prompts), "prompts")

# make_submission(
#     model_q, prompts,
#     out_path=WORK_DIR / "submission_test" / "prediction.json",
#     max_new_tokens=100,        # shorter for the test
#     batch_size=4,
#     temperature=0.0,
#     repetition_penalty=1.1,
# )

In [26]:
# cell 20

@torch.no_grad()
def generate(model: LegalLM, prompts: List[List[int]], cfg: ModelConfig,
             max_new_tokens: int = 300, temperature: float = 0.0,
             top_k: int = 0, top_p: float = 1.0,
             repetition_penalty: float = 1.0, eos_id: Optional[int] = None) -> List[List[int]]:
    """Greedy when temperature == 0. Left-pads the batch."""
    model.eval()
    B = len(prompts)
    L = max(len(p) for p in prompts)
    pad_id = 0

    input_ids = torch.full((B, L), pad_id, dtype=torch.long, device=device)
    attn_mask = torch.zeros((B, L), dtype=torch.long, device=device)
    pos_ids   = torch.zeros((B, L), dtype=torch.long, device=device)
    for i, p in enumerate(prompts):
        input_ids[i, L - len(p):] = torch.tensor(p, device=device)
        attn_mask[i, L - len(p):] = 1
        pos_ids[i, L - len(p):]   = torch.arange(len(p), device=device)

    # --- prefill ---
    logits, _, caches = model(input_ids, attention_mask=attn_mask,
                              position_ids=pos_ids, kv_caches=None)
    next_logits = logits[:, -1, :]

    generated = [[] for _ in range(B)]
    cur_pos = pos_ids[:, -1:] + 1
    eos_hit = [False] * B

    for _ in range(max_new_tokens):
        # ---- sample ----
        lg = next_logits.float()
        if repetition_penalty != 1.0:
            for i in range(B):
                for tid in set(generated[i]):
                    if lg[i, tid] > 0: lg[i, tid] /= repetition_penalty
                    else:              lg[i, tid] *= repetition_penalty
        if temperature == 0.0:
            nxt = lg.argmax(dim=-1)
        else:
            lg = lg / temperature
            if top_k > 0:
                v, _ = torch.topk(lg, top_k, dim=-1)
                lg[lg < v[:, [-1]]] = -float("inf")
            if top_p < 1.0:
                srt, idx = torch.sort(lg, descending=True, dim=-1)
                cdf = srt.softmax(-1).cumsum(-1)
                mask = cdf > top_p
                mask[:, 1:] = mask[:, :-1].clone()
                mask[:, 0] = False
                srt[mask] = -float("inf")
                lg = srt.scatter(-1, idx, srt)
            probs = lg.softmax(-1)
            nxt = torch.multinomial(probs, 1).squeeze(-1)

        for i in range(B):
            if not eos_hit[i]:
                generated[i].append(int(nxt[i].item()))
                if eos_id is not None and int(nxt[i].item()) == eos_id:
                    eos_hit[i] = True

        # ---- decode one step ----
        step_ids = nxt.view(B, 1)
        new_mask = torch.cat([attn_mask, torch.ones(B, 1, device=device, dtype=torch.long)], dim=1)
        logits, _, caches = model(step_ids, attention_mask=new_mask,
                                  position_ids=cur_pos, kv_caches=caches)
        next_logits = logits[:, -1, :]
        attn_mask = new_mask
        cur_pos = cur_pos + 1
    return generated

In [27]:
# Cell 21 — Generation helpers + submission builder

import json, zipfile
from pathlib import Path
from datasets import load_dataset

# ---- 128-word truncation (required by the task) -------------------------
def truncate_words(text: str, n: int = 128) -> str:
    words = text.split()
    return " ".join(words[:n])

# Your tokenizer's decode already uses errors="replace" internally.
def decode_safe(ids):
    return tokenizer.decode(ids)

# ---- Load the test prompts directly from HF -----------------------------
# def load_test_prompts_hf(max_n: int = None) -> Dict[str, str]:
#     """
#     Returns {id_str: prompt_text} for the phase_1 test split.
#     Test rows are keyed by numeric strings: '1'..'1000'.
#     """
#     ds = load_dataset("Exploration-Lab/CS-781-Capstone", "phase_1",
#                       split="test", streaming=True, token=True)
#     out = {}
#     for i, ex in enumerate(ds):
#         # If the ID is inside the row use it, else fall back to the row index.
#         key = ex.get("id") or ex.get("ID")
#         if key is None:
#             key = str(i + 1)     # matches the observed 1..1000 keys
#         out[str(key)] = ex["text"]
#         if max_n is not None and i + 1 >= max_n:
#             break
#     return out

def load_test_prompts_hf(max_n: int = None) -> Dict[str, str]:
    """
    The phase_1 test split is stored as ONE HuggingFace row whose columns are
    the prompt IDs '1', '2', ..., '1000'. Each value is {"text": "..."}.
    """
    ds = load_dataset("Exploration-Lab/CS-781-Capstone", "phase_1",
                      split="test", streaming=True, token=True)
    out = {}
    for row in ds:                                 # typically 1 row
        for k, v in row.items():
            if isinstance(v, dict):
                out[str(k)] = v.get("text", "")
            elif isinstance(v, str):
                out[str(k)] = v
            if max_n is not None and len(out) >= max_n:
                return out
    return out

# ---- Build a Codabench submission zip -----------------------------------
@torch.no_grad()
def make_submission(model, prompts: Dict[str, str], out_path: Path,
                    max_new_tokens: int = 300,
                    batch_size: int = 8,
                    temperature: float = 0.0,
                    top_p: float = 1.0, top_k: int = 0,
                    repetition_penalty: float = 1.1):
    model.eval()
    ids = list(prompts.keys())
    max_prompt = model.cfg.max_seq_len - max_new_tokens
    encoded = []
    for k in ids:
        toks = tokenizer.encode(prompts[k], allow_special=False)
        if len(toks) > max_prompt:
            toks = toks[-max_prompt:]     # left-truncate
        encoded.append(toks)

    order = sorted(range(len(ids)), key=lambda i: len(encoded[i]))
    predictions = {}
    for i in range(0, len(order), batch_size):
        batch = order[i:i + batch_size]
        batch = sorted(batch, key=lambda j: -len(encoded[j]))   # long → short
        outs = generate(model, [encoded[j] for j in batch], model.cfg,
                        max_new_tokens=max_new_tokens,
                        temperature=temperature, top_k=top_k, top_p=top_p,
                        repetition_penalty=repetition_penalty, eos_id=EOS_ID)
        for j, o in zip(batch, outs):
            text = truncate_words(decode_safe(o), 128)
            predictions[ids[j]] = {"text": text}
        print(f"  batch {i//batch_size+1}/{(len(order)+batch_size-1)//batch_size}")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(predictions, ensure_ascii=False))
    print(f"wrote {out_path} ({len(predictions)} predictions)")

    zip_path = out_path.parent / "submission.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_path, arcname="prediction.json")
    print(f"wrote {zip_path}")

In [28]:
# Cell 22 — Generate submission from a trained checkpoint
# =========================================================================
# 1. Load the best checkpoint from the S_baseline run
model, cfg_loaded, extra = load_model(
    WORK_DIR / "runs" / "S_baseline" / "best.pt",
    # WORK_DIR / "runs" / "quick_test" / "best.pt",
    map_location=device,
)
model = model.to(device).eval()
print(f"loaded step {extra.get('step')}, val_loss {extra.get('val_loss')}")

# 2. Pull the 1000 test prompts from HF
prompts = load_test_prompts_hf()
print(f"loaded {len(prompts)} test prompts")
print("sample id:", next(iter(prompts)), "| text:", next(iter(prompts.values()))[:80], "...")

# 3. Generate + zip
make_submission(
    model, prompts,
    out_path=WORK_DIR / "submission" / "prediction.json",
    max_new_tokens=256,
    batch_size=16,
    temperature=0.0,          # greedy
    repetition_penalty=1.1,
)

loaded step 5750, val_loss 2.116826470941305


README.md: 0.00B [00:00, ?B/s]

loaded 1000 test prompts
sample id: 1 | text: The issue involved in these petitions is as to whether the acquisition proceedin ...
  batch 1/63
  batch 2/63
  batch 3/63
  batch 4/63
  batch 5/63
  batch 6/63
  batch 7/63
  batch 8/63
  batch 9/63
  batch 10/63
  batch 11/63
  batch 12/63
  batch 13/63
  batch 14/63
  batch 15/63
  batch 16/63
  batch 17/63
  batch 18/63
  batch 19/63
  batch 20/63
  batch 21/63
  batch 22/63
  batch 23/63
  batch 24/63
  batch 25/63
  batch 26/63
  batch 27/63
  batch 28/63
  batch 29/63
  batch 30/63
  batch 31/63
  batch 32/63
  batch 33/63
  batch 34/63
  batch 35/63
  batch 36/63
  batch 37/63
  batch 38/63
  batch 39/63
  batch 40/63
  batch 41/63
  batch 42/63
  batch 43/63
  batch 44/63
  batch 45/63
  batch 46/63
  batch 47/63
  batch 48/63
  batch 49/63
  batch 50/63
  batch 51/63
  batch 52/63
  batch 53/63
  batch 54/63
  batch 55/63
  batch 56/63
  batch 57/63
  batch 58/63
  batch 59/63
  batch 60/63
  batch 61/63
  batch 62/63
  batch 63

In [29]:
import json, zipfile
sub = json.loads((WORK_DIR / "submission" / "prediction.json").read_text())
print("num predictions:", len(sub))
print("ids present:", len(sub) == 1000)
wc = [len(v["text"].split()) for v in sub.values()]
print("word counts min/mean/max:", min(wc), sum(wc)/len(wc), max(wc))

# Zip structure
with zipfile.ZipFile(WORK_DIR / "submission" / "submission.zip") as z:
    print("zip contents:", z.namelist())   # should be ['prediction.json'] only

num predictions: 1000
ids present: True
word counts min/mean/max: 1 123.804 128
zip contents: ['prediction.json']


In [30]:
# cell 22

In [31]:
# cell 23